In [1]:
!pip install -q openai

In [3]:
OPENAI_API_KEY=""

MODEL = "gpt-4.1-nano"

import os
os.environ["OPENAI_API_KEY"] = OPENAI_API_KEY

In [5]:
#!/usr/bin/env python3
"""
Generate 10 CEFR vocabulary items for a given topic as strict JSON.

Install:
  pip install openai

Env:
  export OPENAI_API_KEY="..."

Run:
  python vocab_json.py --level B2 --topic "election of the first president of the USA"
"""

from __future__ import annotations

import argparse
import json
import os
import re
import sys
from typing import Any, Dict, List

from openai import OpenAI

CEFR_RE = re.compile(r"^(A1|A2|B1|B2|C1|C2)$", re.IGNORECASE)


def validate_level(level: str) -> str:
    level = level.strip().upper()
    if not CEFR_RE.match(level):
        raise ValueError("Invalid level. Use one of: A1, A2, B1, B2, C1, C2.")
    return level


def build_vocab_schema_object(n_words: int = 10) -> Dict[str, Any]:
    """
    Schema for: { "items": [ {word, translation, examples_of_usage}, ... x10 ] }
    Then we print ONLY the array items.
    """
    return {
        "type": "object",
        "additionalProperties": False,
        "properties": {
            "items": {
                "type": "array",
                "minItems": n_words,
                "maxItems": n_words,
                "items": {
                    "type": "object",
                    "additionalProperties": False,
                    "properties": {
                        "word": {"type": "string", "minLength": 1},
                        "translation": {"type": "string", "minLength": 1},  # Ukrainian
                        "examples_of_usage": {
                            "type": "array",
                            "minItems": 2,
                            "maxItems": 3,
                            "items": {"type": "string", "minLength": 1},
                        },
                    },
                    "required": ["word", "translation", "examples_of_usage"],
                },
            }
        },
        "required": ["items"],
    }


def validate_items(items: Any, n_words: int = 10) -> List[Dict[str, Any]]:
    if not isinstance(items, list) or len(items) != n_words:
        raise ValueError(f"Expected 'items' to be a list of exactly {n_words} elements.")

    for i, it in enumerate(items):
        if not isinstance(it, dict):
            raise ValueError(f"Item {i} is not an object.")
        for k in ("word", "translation", "examples_of_usage"):
            if k not in it:
                raise ValueError(f"Item {i} missing key: {k}")
        if not isinstance(it["examples_of_usage"], list) or not (2 <= len(it["examples_of_usage"]) <= 3):
            raise ValueError(f"Item {i} examples_of_usage must have 2–3 sentences.")
        if any(not isinstance(s, str) or not s.strip() for s in it["examples_of_usage"]):
            raise ValueError(f"Item {i} has empty example sentence(s).")

    return items


def try_parse_json(text: str) -> Any:
    # Straight JSON parse
    return json.loads(text)


def generate_vocab(level: str, topic: str, model: str, n_words: int = 10, retries: int = 2) -> List[Dict[str, Any]]:
    client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))

    system = (
        "You generate English vocabulary lists.\n"
        "Translations MUST be Ukrainian.\n"
        "Examples MUST be natural English and appropriate for the requested CEFR level.\n"
        "No profanity. No extra keys. No markdown."
    )

    user = f"""
Create {n_words} English vocabulary items for CEFR level {level} about this topic:

TOPIC: {topic}

Rules:
- Exactly {n_words} items.
- Prefer single words or short terms (1–3 words max).
- Difficulty must match {level}.
- Translation must be Ukrainian.
- Provide 2–3 English example sentences per item.
Return the result as JSON with key "items".
"""

    schema = build_vocab_schema_object(n_words)

    last_err: Exception | None = None

    # ---- Attempt 1: Structured Outputs (json_schema) ----
    for attempt in range(retries + 1):
        try:
            resp = client.responses.create(
                model=model,
                input=[
                    {"role": "system", "content": system},
                    {"role": "user", "content": user},
                ],
                text={
                    "format": {
                        "type": "json_schema",
                        "name": "vocab_list",   # REQUIRED
                        "schema": schema,
                        "strict": True,
                    }
                },
                temperature=0.3,
            )

            parsed = try_parse_json(resp.output_text)
            items = validate_items(parsed.get("items"), n_words)
            return items

        except Exception as e:
            last_err = e
            # strengthen instruction and retry
            user += "\nIMPORTANT: Output MUST be valid JSON and match the schema exactly."
            continue

    # ---- Fallback: JSON mode (json_object) ----
    # This is less strict than json_schema, but often works if schema is unsupported.
    for attempt in range(retries + 1):
        try:
            resp = client.responses.create(
                model=model,
                input=[
                    {"role": "system", "content": system},
                    {"role": "user", "content": user},
                ],
                text={"format": {"type": "json_object"}},
                temperature=0.2,
            )

            parsed = try_parse_json(resp.output_text)
            items = validate_items(parsed.get("items"))
            return items

        except Exception as e:
            last_err = e
            user += f"\nIMPORTANT: Return JSON ONLY, with top-level key 'items' containing exactly {n_words} objects."
            continue

    raise RuntimeError(f"Failed to generate valid output. Last error: {last_err}")



def main() -> None:
    # parser = argparse.ArgumentParser()
    # parser.add_argument("--level", required=True, help="CEFR level: A1, A2, B1, B2, C1, C2")
    # parser.add_argument("--topic", required=True, help="Any topic (can be narrow)")
    # # Use a model that supports Structured Outputs well:
    # parser.add_argument("--model", default="gpt-4o-mini", help="Default: gpt-4o-mini")
    # args = parser.parse_args()



    try:
        # level = validate_level(args.level)
        # topic = args.topic.strip()
        # model = args.model
        level = "C1"
        # topic = "Sillicone Valley HBO Series"
        topic = "Ray Dalio Principles Book"
        model = "gpt-4o-mini"
        n_words = 20

        if not topic:
            raise ValueError("Topic must be a non-empty string.")

        items = generate_vocab(level=level, topic=topic, model=model, n_words=n_words)

        # Print the final JSON array in the exact shape you want
        print(json.dumps(items, ensure_ascii=False, indent=2))

    except Exception as e:
        print(f"ERROR: {e}", file=sys.stderr)
        sys.exit(1)


if __name__ == "__main__":
    main()


[
  {
    "word": "principles",
    "translation": "принципи",
    "examples_of_usage": [
      "His principles guide every decision he makes.",
      "Understanding the principles behind success is crucial.",
      "She wrote down her principles to stay focused."
    ]
  },
  {
    "word": "radical",
    "translation": "радикальний",
    "examples_of_usage": [
      "He proposed a radical change in the company structure.",
      "Radical ideas often challenge the status quo.",
      "The radical approach led to significant improvements."
    ]
  },
  {
    "word": "transparency",
    "translation": "прозорість",
    "examples_of_usage": [
      "Transparency in communication builds trust.",
      "The company's transparency was praised by investors.",
      "She values transparency in her relationships."
    ]
  },
  {
    "word": "accountability",
    "translation": "відповідальність",
    "examples_of_usage": [
      "Accountability is essential for effective teamwork.",
      "He e

In [ ]:
level = "C1"
topic = "Terminator 2 movie"
model = "gpt-4o-mini"

if not topic:
    raise ValueError("Topic must be a non-empty string.")

items = generate_vocab(level=level, topic=topic, model=model)

# Print the final JSON array in the exact shape you want
print(json.dumps(items, ensure_ascii=False, indent=2))

[
  {
    "word": "cyborg",
    "translation": "кіборг",
    "examples_of_usage": [
      "The cyborg in the film represents the fusion of man and machine.",
      "In Terminator 2, the cyborg is programmed to protect rather than destroy.",
      "Cyborgs challenge our understanding of humanity."
    ]
  },
  {
    "word": "apocalypse",
    "translation": "апокаліпсис",
    "examples_of_usage": [
      "The movie depicts a future on the brink of apocalypse.",
      "Many films explore the theme of apocalypse and its consequences.",
      "In Terminator 2, the threat of apocalypse looms over humanity."
    ]
  },
  {
    "word": "sentient",
    "translation": "сентиментальний",
    "examples_of_usage": [
      "The sentient machines in the film raise ethical questions.",
      "As technology advances, the idea of sentient AI becomes more plausible.",
      "The sentient nature of the cyborg adds depth to its character."
    ]
  },
  {
    "word": "dystopia",
    "translation": "дистопія

backend service on Flask (domain https://diploma--teamerking.replit.app)


In [ ]:
from flask import Flask, jsonify, request
import prompt as pt

app = Flask(__name__)

@app.route("/")
def root():
    return jsonify(status="healthy"), 200

@app.route("/getWords", methods=["POST"])
def getwords():
    data = request.get_json()
    level:str = data.get("level")
    topic:str = data.get("topic")

    list = pt.generate_vocab(level=level, topic=topic, model="gpt-4o-mini")
    return jsonify(words=list), 200

if __name__ == "__main__":
    app.run(host="0.0.0.0", port=5000)
